# 🌊 UNIFIED RWKV-FLOOD SCREENING & STRUCTURAL INTEGRATION BENCHMARK
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/doantrongthai/rwkv-flood/blob/main/screening_rwkv_architectures.ipynb)

### Size 256x256 | Batch Size 4 | Lovasz-Focal Loss | Automatic NaN-Skip Protection

This notebook executes a systematic evaluation of RWKV linear-attention integration across:
1. **Existing Literature Baselines**: `vrwkv`, `rsrwkv`, `scrwkv`, `u_rwkv`, `seg_rwkv`
2. **Single-Component Integration**:
   - `rwkv_enc` (Encoder only), `rwkv_bot` (Bottleneck only), `rwkv_dec` (Decoder only), `rwkv_skip` (Skip only)
3. **Pairwise 2-Component Combinations**:
   - `rwkv_enc_bot`, `rwkv_bot_dec`, `rwkv_enc_dec`, `rwkv_enc_skip`, `rwkv_bot_skip`, `rwkv_dec_skip`
4. **Triple 3-Component Combinations**:
   - `rwkv_enc_bot_dec`, `rwkv_enc_bot_skip`, `rwkv_enc_dec_skip`, `rwkv_bot_dec_skip`
5. **Full Quad Integration & Domain Specialized**:
   - `rwkv_all4` (Encoder + Bottleneck + Decoder + Skip simultaneously)
   - `hydrorwkv` (Directional Strip Convolutions 1x7 & 7x1 + Bi-WKV)

**🛡️ Built-in NaN Early-Exit Shield**:
If any unstable baseline or configuration encounters non-finite loss (`NaN`/`Inf`), the execution loop **immediately aborts that model and skips to the next**, ensuring the pipeline never halts or hangs.

In [ ]:
# 1. Clone RWKV-Flood repository and install dependencies
!git clone https://github.com/doantrongthai/rwkv-flood.git
%cd rwkv-flood
!pip install -r requirements.txt
!pip install -q tabulate fvcore thop pandas

In [ ]:
# 2. Check GPU Environment & CUDA Acceleration
import torch
print(f"PyTorch Version : {torch.__version__}")
print(f"CUDA Available  : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device Name     : {torch.cuda.get_device_name(0)}")
    print(f"Compute Capability: {torch.cuda.get_device_capability(0)}")

In [ ]:
# 3. Ensure Datasets are Downloaded
import subprocess, sys

for ds in ["floodkaggle", "aifloodsense"]:
    print(f"Ensuring dataset '{ds}' is prepared...")
    subprocess.run([sys.executable, "main.py", "--dataset", ds, "--download", "--epochs", "1", "--batch_size", "2", "--model", "unet", "--size", "256"], check=False)

print("✓ Datasets ready in data/ directory.")

In [ ]:
# 4. Master Architecture Registry & Experiment Configuration
IMAGE_SIZE = 256
BATCH_SIZE = 4
EPOCHS = 100           # Screening epochs (enforcing standard 100 epochs)
LR = 0.001
LOSS_NAME = "lovasz_focal"
NUM_CLASSES = 1
SEED = 42
DATASETS = ["floodkaggle", "aifloodsense"]

# 1. Literature Baselines
LITERATURE_MODELS = [
    "vrwkv",          # Vision-RWKV (ICLR 2025 Spotlight) - known stable
    "rsrwkv",         # RSRWKV (IEEE TGRS 2025)
    "scrwkv",         # SCRWKV (ICML 2026)
    "u_rwkv",         # U-RWKV (2025)
    "seg_rwkv",       # SegRWKV (JCDE 2024/2025)
]

# 2. Systematic Single Component Integration
SINGLE_MODELS = [
    "rwkv_enc",       # RWKV in Encoder only
    "rwkv_bot",       # RWKV in Bottleneck only
    "rwkv_dec",       # RWKV in Decoder only
    "rwkv_skip",      # RWKV in Skip Connections only
]

# 3. Pairwise 2-Component Combinations
PAIR_MODELS = [
    "rwkv_enc_bot",   # Encoder + Bottleneck
    "rwkv_bot_dec",   # Bottleneck + Decoder
    "rwkv_enc_dec",   # Encoder + Decoder
    "rwkv_enc_skip",  # Encoder + Skip
    "rwkv_bot_skip",  # Bottleneck + Skip
    "rwkv_dec_skip",  # Decoder + Skip
]

# 4. Triple 3-Component Combinations
TRIPLE_MODELS = [
    "rwkv_enc_bot_dec",   # Encoder + Bottleneck + Decoder
    "rwkv_enc_bot_skip",  # Encoder + Bottleneck + Skip
    "rwkv_enc_dec_skip",  # Encoder + Decoder + Skip
    "rwkv_bot_dec_skip",  # Bottleneck + Decoder + Skip
]

# 5. Full Quad Integration & Domain Specialized
FULL_MODELS = [
    "rwkv_all4",      # All 4 components simultaneously (E + B + D + S)
    "hydrorwkv",      # Directional Strip Convolutions (1x7 & 7x1) + Bi-WKV
]

ALL_RWKV_MODELS = LITERATURE_MODELS + SINGLE_MODELS + PAIR_MODELS + TRIPLE_MODELS + FULL_MODELS

print(f"Total architectures registered: {len(ALL_RWKV_MODELS)}")
print(f"Literature Baselines: {len(LITERATURE_MODELS)}")
print(f"Systematic Structural Exploration: {len(SINGLE_MODELS) + len(PAIR_MODELS) + len(TRIPLE_MODELS) + len(FULL_MODELS)}")
print(f"Configuration: Batch={BATCH_SIZE}, Size={IMAGE_SIZE}, Epochs={EPOCHS}, Loss={LOSS_NAME}, LR={LR}")

In [ ]:
# 5. Automated Hardware Profiling Table (Params, GFLOPs, Latency, FPS)
import pandas as pd
import torch
from models import get_model
from utils.metrics import calculate_model_complexity, measure_inference_time

device = "cuda" if torch.cuda.is_available() else "cpu"
profiling_records = []
print("=" * 90)
print(f"{'ARCH':<20} | {'CATEGORY':<14} | {'PARAMS':<10} | {'GFLOPS':<8} | {'LATENCY':<10} | {'FPS':<8}")
print("=" * 90)

for model_name in ALL_RWKV_MODELS:
    try:
        model = get_model(model_name, num_classes=NUM_CLASSES, input_size=IMAGE_SIZE, seed=SEED).to(device)
        model.eval()
        complexity = calculate_model_complexity(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)
        perf = measure_inference_time(model, input_size=(1, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)
        
        category = "Literature" if model_name in LITERATURE_MODELS else ("Specialized" if model_name == "hydrorwkv" else "Exploration")
        p_val = complexity.get("deployable_params", complexity.get("total_params", complexity.get("params", 0)))
        g_val = complexity.get("gflops", 0.0)
        lat_val = perf.get("latency_ms", perf.get("latency_mean_ms", 0.0))
        fps_val = perf.get("fps", 0.0)

        profiling_records.append({
            "Architecture": model_name,
            "Category": category,
            "Params": p_val,
            "GFLOPs": g_val,
            "Latency (ms)": lat_val,
            "FPS": fps_val
        })
        print(f"{model_name:<20} | {category:<14} | {p_val:>10,d} | {g_val:>8.3f} | {lat_val:>8.2f} ms | {fps_val:>8.1f}")
        del model
        torch.cuda.empty_cache()
    except Exception as e:
        print(f"{model_name:<20} | FAILED PROFILING: {e}")

df_profile = pd.DataFrame(profiling_records)
df_profile.to_csv("rwkv_hardware_profiling.csv", index=False)
print("=" * 90)
print("✓ Hardware profiling saved to rwkv_hardware_profiling.csv")

In [ ]:
# 6. UNIFIED SEQUENTIAL SCREENING WITH AUTOMATIC NAN-SKIP PROTECTION
import re, subprocess, os, sys

screening_results = {}

for idx, model_name in enumerate(ALL_RWKV_MODELS, 1):
    category = "Literature" if model_name in LITERATURE_MODELS else "Exploration"
    print(f"\n{'#'*90}")
    print(f"[{idx}/{len(ALL_RWKV_MODELS)}] BENCHMARKING: {model_name.upper()} ({category})")
    print(f"{'#'*90}")
    screening_results[model_name] = {}
    model_aborted = False

    for ds in DATASETS:
        if model_aborted:
            print(f"--> [SKIP] Model {model_name} previously aborted due to NaN. Skipping {ds.upper()}...")
            screening_results[model_name][ds] = {"mIoU": None, "Dice": None, "Status": "SKIPPED_NAN"}
            continue

        print(f"\n>>> Training {model_name} on {ds.upper()} (Seed {SEED}) <<<")
        cmd = [
            sys.executable, "main.py",
            "--dataset", ds,
            "--model", model_name,
            "--num_classes", str(NUM_CLASSES),
            "--loss", LOSS_NAME,
            "--size", str(IMAGE_SIZE),
            "--epochs", str(EPOCHS),
            "--batch_size", str(BATCH_SIZE),
            "--lr", str(LR),
            "--seed", str(SEED),
            "--download"
        ]

        proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        full_log = []
        is_nan_detected = False

        for line in proc.stdout:
            full_log.append(line)
            if "Epoch [" in line or "mIoU:" in line or "Best" in line or "EVALUATION" in line or "ALERT" in line or "FAILED_NAN" in line:
                print(line, end="")
            if "[ALERT] NaN" in line or "FAILED_NAN" in line:
                is_nan_detected = True

        proc.wait()
        log_str = "".join(full_log)

        if is_nan_detected or "FAILED_NAN" in log_str:
            print(f"\n[!] DETECTED NON-FINITE DIVERGENCE (NaN/Inf) IN {model_name.upper()}! SKIPPING MODEL...")
            model_aborted = True
            screening_results[model_name][ds] = {"mIoU": None, "Dice": None, "Status": "FAILED_NAN"}
            continue

        # Parse metrics using regex
        miou = None
        dice = None
        loss = None

        miou_match = re.findall(r"mIoU[:=]\s*([0-9.]+)", log_str)
        if miou_match:
            miou = float(miou_match[-1])

        dice_match = re.findall(r"(?:Dice|F1)[:=]\s*([0-9.]+)", log_str)
        if dice_match:
            dice = float(dice_match[-1])

        loss_match = re.findall(r"(?:Val Loss|Loss)[:=]\s*([0-9.]+)", log_str)
        if loss_match:
            loss = float(loss_match[-1])

        screening_results[model_name][ds] = {
            "mIoU": miou,
            "Dice": dice,
            "Val_Loss": loss,
            "Status": "COMPLETED"
        }
        print(f"--> [{model_name} on {ds}] Completed: mIoU={miou}, Dice={dice}, Val_Loss={loss}")

print("\n✓ All runs finished! Summary table compiling below...")

In [ ]:
# 7. SUMMARY LEADERBOARD & STRUCTURAL INTEGRATION ANALYSIS
import pandas as pd
from tabulate import tabulate

master_rows = []
for m in ALL_RWKV_MODELS:
    fk = screening_results.get(m, {}).get("floodkaggle", {})
    afs = screening_results.get(m, {}).get("aifloodsense", {})

    fk_status = fk.get("Status", "N/A")
    afs_status = afs.get("Status", "N/A")
    overall_status = "FAILED_NAN" if (fk_status == "FAILED_NAN" or afs_status == "FAILED_NAN") else "COMPLETED"

    fk_miou = fk.get("mIoU")
    afs_miou = afs.get("mIoU")
    mean_miou = ((fk_miou or 0.0) + (afs_miou or 0.0)) / 2.0 if (fk_miou is not None and afs_miou is not None) else None

    fk_dice = fk.get("Dice")
    afs_dice = afs.get("Dice")
    mean_dice = ((fk_dice or 0.0) + (afs_dice or 0.0)) / 2.0 if (fk_dice is not None and afs_dice is not None) else None

    prof_row = df_profile[df_profile["Architecture"] == m]
    params = prof_row["Params"].values[0] if len(prof_row) > 0 else 0
    gflops = prof_row["GFLOPs"].values[0] if len(prof_row) > 0 else 0.0
    fps = prof_row["FPS"].values[0] if len(prof_row) > 0 else 0.0
    category = prof_row["Category"].values[0] if len(prof_row) > 0 else "Exploration"

    master_rows.append({
        "Architecture": m,
        "Category": category,
        "Status": overall_status,
        "Params": params,
        "GFLOPs": gflops,
        "FPS": fps,
        "FloodKaggle mIoU": f"{fk_miou:.4f}" if fk_miou is not None else "NaN",
        "AIFloodSense mIoU": f"{afs_miou:.4f}" if afs_miou is not None else "NaN",
        "Mean mIoU": f"{mean_miou:.4f}" if mean_miou is not None else "NaN",
        "Mean Dice": f"{mean_dice:.4f}" if mean_dice is not None else "NaN"
    })

df_master = pd.DataFrame(master_rows)
df_master.to_csv("master_rwkv_screening_results.csv", index=False)

# -------------------------------------------------------------
# TABLE 1: COMPLETE RWKV BENCHMARK LEADERBOARD
# -------------------------------------------------------------
df_sorted = df_master.sort_values(
    by=["Status", "Mean mIoU"],
    ascending=[True, False]
).reset_index(drop=True)
df_sorted.index += 1

print("=" * 115)
print("TABLE 1: COMPLETE RWKV BENCHMARK LEADERBOARD (Size 256x256, Seed 42)")
print("=" * 115)
print(tabulate(df_sorted, headers="keys", tablefmt="github", showindex=True))

In [ ]:
# 8. Package Results & Download Archive
import shutil, os
from google.colab import files

os.makedirs("results_bundle", exist_ok=True)
for f in ["rwkv_hardware_profiling.csv", "master_rwkv_screening_results.csv"]:
    if os.path.exists(f):
        shutil.copy(f, os.path.join("results_bundle", f))

if os.path.exists("outputs"):
    for item in os.listdir("outputs"):
        p = os.path.join("outputs", item)
        if os.path.isfile(p) and item.endswith((".pth", ".csv", ".json")):
            shutil.copy(p, os.path.join("results_bundle", item))

shutil.make_archive("rwkv_screening_results", "zip", "results_bundle")
print("✓ Results archived to rwkv_screening_results.zip")
files.download("rwkv_screening_results.zip")